# 17 · Malha existente: grade de janelas de calibração e de conferência

> **Regra da disciplina (aula 3):** especificação FOCUS → código gerado pelo assistente → executar → *gate* contra uma resposta já conhecida → aceitar.
> **Quando o gate falha, a restrição que faltou entra em `U`. O código não é remendado.**
>
> Cada passo abaixo tem três partes: a especificação FOCUS enviada ao assistente, o gate declarado **antes** do código (com a fonte da resposta conhecida e da tolerância) e o código que voltou, executado e gateado. O registro de uso de IA é exportado ao final (`registro_uso_ia.csv`).

**Pedido do autor (16/11).** Varrer janelas de 1, 2, 5 e 10 anos, do início, do meio e do fim do período, para frente e para trás, e ver em que cenário o EAR calibrado identifica melhor os quilômetros que tiveram descarrilamento por via. Sempre na malha existente, onde o acidente real permite conferir.

**Desenho (declarado antes dos resultados).**

1. Janelas adjacentes: calibração de 1, 2, 5 ou 10 anos; conferência de 1, 2 ou 5 anos imediatamente depois (para frente) ou imediatamente antes (para trás); todas as posições possíveis entre 2011 e 2025.
2. Todos os anos disponíveis: calibração com todos os anos anteriores (mínimo de 3) e conferência nos 1, 2 ou 5 anos seguintes; e o espelho para trás.
3. Com defasagem: calibração com todos os anos anteriores e conferência em um único ano, 3 ou 5 anos depois do fim da calibração; e o espelho para trás.

Em todos: nota só com atributos da via (regressão logística do notebook 16, hiperparâmetros fixos); duas conferências (mesmos km; km fora da calibração, por grupos de trecho); regra do EAR por faixas como baseline na mesma conferência. Todos os cenários são reportados; o melhor cenário é descrito, não escolhido como resultado principal.

**Gates.** 2.681 descarrilamentos por via; janelas disjuntas em todos os cenários; teste de vazamento pela média de 20 embaralhamentos entre 0,45 e 0,55.

In [1]:
# === Setup: caminhos, bibliotecas e as três ferramentas (gate, gate_between, log_ia) ===
import os, sys, json, re, unicodedata, datetime as dt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if os.environ.get("ET261400_BASE"):
    BASE = os.environ["ET261400_BASE"]                # definido pelo notebook 00 (execução em lote) ou pela execução local
elif IN_COLAB and os.path.exists("/content/ET261400/data"):
    BASE = "/content/ET261400"                        # pacote já descompactado na sessão do Colab
elif IN_COLAB and os.path.exists("/content/et261400_colab.zip"):
    import zipfile
    zipfile.ZipFile("/content/et261400_colab.zip").extractall("/content/ET261400")   # pacote enviado pelo painel de arquivos
    BASE = "/content/ET261400"
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = "/content/drive/MyDrive/ET261400"          # pasta do projeto no Drive (ajustar se for outra)
else:
    BASE = os.environ.get("ET261400_BASE", os.path.abspath(".."))   # execução local: raiz do repositório
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
FIG, TAB = f"{BASE}/results/figures", f"{BASE}/results/tables"
for d in (PROC, FIG, TAB):
    os.makedirs(d, exist_ok=True)
print("BASE =", BASE)

_GATES = []          # todo gate executado fica registrado aqui
_LOG_IA = []         # registro de uso de IA: passo, ferramenta, especificação, resultado

def gate(name, got, expected, tol=0.0, source=""):
    """Compara um número calculado com um número que já se tinha antes do código.
    tol vem da fonte (meio passo de arredondamento, faixa publicada), nunca é alargada até passar."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = abs(got - expected) <= tol
    _GATES.append(dict(gate=name, got=got, expected=expected, tol=tol, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, esperado {expected} (tol +/-{tol}) {('| ' + source) if source else ''}")
    return ok

def gate_between(name, got, low, high, source=""):
    """O mesmo, quando a fonte publica uma faixa e não um número."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = low <= got <= high
    _GATES.append(dict(gate=name, got=got, expected=f"[{low}, {high}]", tol=None, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, faixa [{low}, {high}] {('| ' + source) if source else ''}")
    return ok

def log_ia(passo, ferramenta, especificacao, resultado, verificacao):
    """Registro de uso de IA exigido pela disciplina: o que foi pedido, a que ferramenta, o que voltou e como foi verificado."""
    _LOG_IA.append(dict(data=dt.date.today().isoformat(), passo=passo, ferramenta=ferramenta,
                        especificacao=especificacao.strip(), resultado=resultado, verificacao=verificacao))

def exportar_registros(prefixo):
    pd.DataFrame(_GATES).to_csv(f"{TAB}/{prefixo}_gates.csv", index=False)
    pd.DataFrame(_LOG_IA).to_csv(f"{TAB}/{prefixo}_registro_uso_ia.csv", index=False)
    n_ok = sum(g["ok"] for g in _GATES)
    print(f"gates: {n_ok}/{len(_GATES)} PASS | registros de IA: {len(_LOG_IA)} | exportados em {TAB}")

def norm_txt(s):
    """Maiúsculas sem acento, espaços simples: chave de junção para nomes de pátios e estações."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).upper().strip()


BASE = /home/claude


In [2]:
import warnings; warnings.filterwarnings("ignore")
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from joblib import Parallel, delayed
SEED = 42
C0 = pd.read_csv(f"{PROC}/11_celulas_1km.csv")
C = C0[~C0["sentido"].astype(str).str.contains("Interromp")].sort_values(["ferrovia", "linha", "km_a"]).reset_index(drop=True); gate("células em operação", len(C), 20936, source="notebook 11")
E = pd.read_csv(f"{PROC}/11_eventos_celulas.csv"); E = E[E["cel_id"].isin(C["cel_id"])].copy(); E["ano"] = E["ano"].astype(int); EV = E[E["via"]]
gate("descarrilamentos por via nas células", len(EV), 2681, source="notebook 11")
idx = pd.Series(np.arange(len(C)), index=C["cel_id"]); ANOS = list(range(2011, 2026))
CA = np.zeros((len(ANOS), len(C)))
for k, a in enumerate(ANOS):
    s = EV[EV["ano"] == a].groupby("cel_id").size(); CA[k, idx[s.index].values] = s.values
gate("soma da matriz ano x célula", int(CA.sum()), 2681, source="partição por ano")
def contagem(a0, a1): return CA[a0 - 2011:a1 - 2011 + 1].sum(0)
DIST = ["dmin_patio", "dmin_amv", "frac_zona_patio"] + [f"f16p{j}" for j in range(5)] + [f"f16a{j}" for j in range(5)]
CURV = ["rmin_geo"] + [f"fr_lt{l}" for l in (300, 400, 500, 600, 800)]
NUM = ["rampa_max_pct", "carga_eixo_t", "taxa_dorm", "gab_vert_m", "gab_horiz_m", "cap_vinc", "dias_op", "raio_min_m", "vma_carregado"]
CAT = ["dormente", "perfil_trilho", "fixacao", "sinalizacao", "controle_circ", "bitola", "sentido", "n_linhas"]
for c in NUM + DIST + CURV: C[c] = pd.to_numeric(C[c], errors="coerce")
for c in CAT: C[c] = C[c].astype(str).str.strip().replace({"nan": "NA", "": "NA"})
def design(df, num, cat):
    X = df[num].copy(); med = X.median().fillna(0.0)
    for c in num: X[c + "_na"] = X[c].isna().astype(float); X[c] = X[c].fillna(med[c])
    return pd.concat([X, pd.get_dummies(df[cat], dtype=float)], axis=1).values
X = StandardScaler().fit_transform(design(C, DIST + CURV + NUM, CAT))  # padronização só usa atributos, nunca a resposta
LN = C["len_km"].values; fer = np.asarray(C["ferrovia"], dtype=object); FERC = pd.factorize(C["ferrovia"])[0]; SEGC = pd.factorize(C["seg_id"].astype(str))[0]; NSEG = SEGC.max() + 1
FOLD = np.random.default_rng(SEED).permutation(NSEG)[SEGC] % 5; DES = np.random.default_rng(SEED).random(len(C))
def cls16(d): return np.select([d <= 300, d <= 600, d <= 1200, d <= 1800], [4, 3, 2, 1], 0)
def rcls16(r_): return np.select([r_ < 500, r_ < 1200, r_ < 2300], [3, 2, 1], 0)
B2 = (0.5 * cls16(C["dmin_patio"].fillna(1e9)) / 4 + 0.5 * rcls16(C["rmin_geo"].fillna(1e6)) / 3).astype(float)
def auc(yy, s):
    n1 = yy.sum(); n0_ = len(yy) - n1; return float((rankdata(s)[yy == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0_)) if n1 >= 5 and n0_ >= 5 else np.nan
def auc_dentro(s, yy):
    a, w = [], []
    for f in range(FERC.max() + 1):
        i = FERC == f
        if yy[i].sum() >= 5 and yy[i].sum() < i.sum(): a.append(auc(yy[i], s[i])); w.append(yy[i].sum())
    return float(np.average(a, weights=w)) if a else np.nan
def captura(s, evv, p=0.20):
    o = np.lexsort((DES, -s)); cum = np.cumsum(LN[o]) / LN.sum(); return float(evv[o][cum <= p].sum() / evv.sum())
def lr(Xtr, ytr, Xte): return LogisticRegression(C=0.5, max_iter=3000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
def cenario(tipo, cal, conf, embaralhar=None):
    assert cal[1] < conf[0] or conf[1] < cal[0], "janelas sobrepostas"
    ycal = (contagem(*cal) > 0).astype(int); cconf = contagem(*conf); yconf = (cconf > 0).astype(int)
    if ycal.sum() < 20 or yconf.sum() < 20: return []
    if embaralhar is not None: ycal = np.random.default_rng(1000 + embaralhar).permutation(ycal)
    base = dict(tipo=tipo, cal_ini=cal[0], cal_fim=cal[1], conf_ini=conf[0], conf_fim=conf[1], anos_calibracao=cal[1] - cal[0] + 1, anos_conferencia=conf[1] - conf[0] + 1, sentido="para frente" if cal[1] < conf[0] else "para trás",
                defasagem=(conf[0] - cal[1] - 1) if cal[1] < conf[0] else (cal[0] - conf[1] - 1), km_com_acidente_calibracao=int(ycal.sum()), km_com_acidente_conferido=int(yconf.sum()))
    def linha(nome, conferencia, s): return dict(base, metodo=nome, conferencia=conferencia, auc=auc(yconf, s), auc_dentro=auc_dentro(s, yconf), captura20=captura(s, cconf))
    s1 = lr(X, ycal, X); s2 = np.zeros(len(C))
    for k in range(5): te = FOLD == k; s2[te] = lr(X[~te], ycal[~te], X[te])
    return [linha("Regra do EAR por faixas", "mesmos km", B2), linha("Regra do EAR por faixas", "km fora da calibração", B2), linha("EAR calibrado (atributos da via)", "mesmos km", s1), linha("EAR calibrado (atributos da via)", "km fora da calibração", s2)]
J = []
for L_ in (1, 2, 5, 10):
    for M_ in (1, 2, 5):
        for a in range(2011, 2026 - L_ - M_ + 1): J.append(("janelas adjacentes", (a, a + L_ - 1), (a + L_, a + L_ + M_ - 1)))
        for a in range(2011 + M_, 2026 - L_ + 1): J.append(("janelas adjacentes", (a, a + L_ - 1), (a - M_, a - 1)))
for M_ in (1, 2, 5):
    for T in range(2014, 2026 - M_ + 1): J.append(("todos os anos disponíveis", (2011, T - 1), (T, T + M_ - 1)))
    for T in range(2010 + M_, 2023): J.append(("todos os anos disponíveis", (T + 1, 2025), (T - M_ + 1, T)))
for g in (3, 5):
    for T in range(2014, 2026 - g + 1): J.append(("com defasagem", (2011, T - 1), (T + g - 1, T + g - 1)))
    for T in range(2010 + g, 2023): J.append(("com defasagem", (T + 1, 2025), (T - g + 1, T - g + 1)))
print("cenários declarados:", len(J)); gate_between("número de cenários", float(len(J)), 250, 400, source="desenho")
log_ia("17.1 grade declarada", "Claude", "FOCUS: grade de janelas adjacentes, todos os anos e defasagem; para frente e para trás; só atributos da via", "lista J", "3 gates de contagem; gate do número de cenários")

[PASS] células em operação: obtido 20936, esperado 20936 (tol +/-0.0) | notebook 11
[PASS] descarrilamentos por via nas células: obtido 2681, esperado 2681 (tol +/-0.0) | notebook 11
[PASS] soma da matriz ano x célula: obtido 2681, esperado 2681 (tol +/-0.0) | partição por ano
cenários declarados: 310
[PASS] número de cenários: obtido 310.0, faixa [250, 400] | desenho


In [3]:
import time; t0 = time.time()
OUT = Parallel(n_jobs=2, batch_size=4)(delayed(cenario)(*j) for j in J); R = pd.DataFrame([l for o in OUT for l in o]); print("minutos:", round((time.time() - t0) / 60, 1), "| linhas:", len(R), "| cenários avaliados:", len(R) // 4)
R.to_csv(f"{TAB}/17_grade_por_cenario.csv", index=False)
vz = [cenario("vazamento", (2011, 2015), (2021, 2025), embaralhar=k) for k in range(20)]; vzm = pd.Series([l["auc"] for o in vz for l in o if l["metodo"].startswith("EAR calibrado") and l["conferencia"] == "km fora da calibração"])
print("vazamento: média", round(vzm.mean(), 3), "| mínimo", round(vzm.min(), 3), "| máximo", round(vzm.max(), 3)); gate_between("teste de vazamento (média de 20 embaralhamentos)", float(vzm.mean()), 0.45, 0.55, source="acaso")
M = ["auc", "auc_dentro", "captura20"]
print("== Geral: todos os cenários =="); g0 = R.groupby(["metodo", "conferencia"], sort=False)[M].agg(["mean", "min", "max"]).round(3); display(g0)
print("== Por tipo e sentido =="); g1 = R.groupby(["tipo", "sentido", "metodo", "conferencia"], sort=False)[M].mean().round(3); g1["n"] = R.groupby(["tipo", "sentido", "metodo", "conferencia"], sort=False).size(); display(g1)
A = R[(R.tipo == "janelas adjacentes") & (R.metodo.str.startswith("EAR calibrado"))]
print("== Janelas adjacentes: AUC por anos de calibração (linhas) e de conferência (colunas) ==")
for conf in ("mesmos km", "km fora da calibração"):
    for sen in ("para frente", "para trás"): print(conf, "|", sen); display(A[(A.conferencia == conf) & (A.sentido == sen)].pivot_table(index="anos_calibracao", columns="anos_conferencia", values="auc").round(3))
print("== AUC por ano final da janela conferida (para frente, km fora da calibração) =="); display(R[(R.sentido == "para frente") & (R.conferencia == "km fora da calibração")].pivot_table(index="conf_fim", columns="metodo", values="auc").round(3))
res = R.groupby(["tipo", "sentido", "anos_calibracao", "anos_conferencia", "defasagem", "metodo", "conferencia"], sort=False)[M].agg(["mean", "min", "max", "count"]).round(3); res.to_csv(f"{TAB}/17_grade_resumo.csv")
gan = R.pivot_table(index=["tipo", "cal_ini", "cal_fim", "conf_ini", "conf_fim", "conferencia"], columns="metodo", values="auc"); d = gan["EAR calibrado (atributos da via)"] - gan["Regra do EAR por faixas"]
print("EAR calibrado acima da regra por faixas em", int((d > 0).sum()), "de", len(d), "comparações; diferença média", round(d.mean(), 3), "| mínima", round(d.min(), 3), "| máxima", round(d.max(), 3))
top = R[R.metodo.str.startswith("EAR calibrado") & (R.conferencia == "km fora da calibração")].sort_values("auc", ascending=False); print("== Dez melhores e dez piores cenários (km fora da calibração) =="); display(top.head(10)[["tipo", "cal_ini", "cal_fim", "conf_ini", "conf_fim", "km_com_acidente_conferido", "auc", "auc_dentro", "captura20"]]); display(top.tail(10)[["tipo", "cal_ini", "cal_fim", "conf_ini", "conf_fim", "km_com_acidente_conferido", "auc", "auc_dentro", "captura20"]])
log_ia("17.2 grade executada", "Claude", "FOCUS: todos os cenários reportados; melhor cenário descrito, não escolhido como resultado principal", "17_grade_*.csv", "gate de vazamento por média de 20 embaralhamentos")
exportar_registros("17")

minutos: 2.5 | linhas: 1240 | cenários avaliados: 310


vazamento: média 0.48 | mínimo 0.395 | máximo 0.619
[PASS] teste de vazamento (média de 20 embaralhamentos): obtido 0.48002879246904406, faixa [0.45, 0.55] | acaso
== Geral: todos os cenários ==


auc                \
                                                         mean    min    max   
metodo                           conferencia                                  
Regra do EAR por faixas          mesmos km              0.587  0.554  0.631   
                                 km fora da calibração  0.587  0.554  0.631   
EAR calibrado (atributos da via) mesmos km              0.750  0.661  0.853   
                                 km fora da calibração  0.722  0.648  0.822   

                                                       auc_dentro         \
                                                             mean    min   
metodo                           conferencia                               
Regra do EAR por faixas          mesmos km                  0.584  0.550   
                                 km fora da calibração      0.584  0.550   
EAR calibrado (atributos da via) mesmos km                  0.671  0.616   
                                 km fora da calibração      0.626  0.519   

                                                              captura20  \
                                                          max      mean   
metodo                           conferencia                              
Regra do EAR por faixas          mesmos km              0.633     0.290   
                                 km fora da calibração  0.633     0.290   
EAR calibrado (atributos da via) mesmos km              0.717     0.530   
                                 km fora da calibração  0.684     0.481   

                                                                      
                                                          min    max  
metodo                           conferencia                          
Regra do EAR por faixas          mesmos km              0.217  0.364  
                                 km fora da calibração  0.217  0.364  
EAR calibrado (atributos da via) mesmos km              0.361  0.732  
                                 km fora da calibração  0.328  0.714

== Por tipo e sentido ==


auc  \
tipo                      sentido     metodo                           conferencia                    
janelas adjacentes        para frente Regra do EAR por faixas          mesmos km              0.584   
                                                                       km fora da calibração  0.584   
                                      EAR calibrado (atributos da via) mesmos km              0.761   
                                                                       km fora da calibração  0.731   
                          para trás   Regra do EAR por faixas          mesmos km              0.590   
                                                                       km fora da calibração  0.590   
                                      EAR calibrado (atributos da via) mesmos km              0.747   
                                                                       km fora da calibração  0.715   
todos os anos disponíveis para frente Regra do EAR por faixas          mesmos km              0.586   
                                                                       km fora da calibração  0.586   
                                      EAR calibrado (atributos da via) mesmos km              0.746   
                                                                       km fora da calibração  0.727   
                          para trás   Regra do EAR por faixas          mesmos km              0.590   
                                                                       km fora da calibração  0.590   
                                      EAR calibrado (atributos da via) mesmos km              0.744   
                                                                       km fora da calibração  0.718   
com defasagem             para frente Regra do EAR por faixas          mesmos km              0.577   
                                                                       km fora da calibração  0.577   
                                      EAR calibrado (atributos da via) mesmos km              0.743   
                                                                       km fora da calibração  0.727   
                          para trás   Regra do EAR por faixas          mesmos km              0.593   
                                                                       km fora da calibração  0.593   
                                      EAR calibrado (atributos da via) mesmos km              0.723   
                                                                       km fora da calibração  0.701   

                                                                                              auc_dentro  \
tipo                      sentido     metodo                           conferencia                         
janelas adjacentes        para frente Regra do EAR por faixas          mesmos km                   0.580   
                                                                       km fora da calibração       0.580   
                                      EAR calibrado (atributos da via) mesmos km                   0.676   
                                                                       km fora da calibração       0.626   
                          para trás   Regra do EAR por faixas          mesmos km                   0.588   
                                                                       km fora da calibração       0.588   
                                      EAR calibrado (atributos da via) mesmos km                   0.670   
                                                                       km fora da calibração       0.626   
todos os anos disponíveis para frente Regra do EAR por faixas          mesmos km                   0.582   
                                                                       km fora da calibração       0.582   
                                      EAR calibrado (atributos da via) mesmos km                   0.669   
             

== Janelas adjacentes: AUC por anos de calibração (linhas) e de conferência (colunas) ==
mesmos km | para frente


... (9 outputs left out)

EAR calibrado acima da regra por faixas em 620 de 620 comparações; diferença média 0.149 | mínima 0.05 | máxima 0.28
== Dez melhores e dez piores cenários (km fora da calibração) ==


,tipo,cal_ini,cal_fim,conf_ini,conf_fim,km_com_acidente_conferido,auc,auc_dentro,captura20
399,janelas adjacentes,2024,2025,2023,2023,111,0.821998,0.657323,0.714286
607,janelas adjacentes,2020,2024,2025,2025,55,0.797488,0.615002,0.561404
599,janelas adjacentes,2018,2022,2023,2023,111,0.797375,0.616399,0.616071
787,janelas adjacentes,2015,2024,2025,2025,55,0.796553,0.650638,0.543860
107,janelas adjacentes,2024,2024,2023,2023,111,0.793424,0.630398,0.660714
495,janelas adjacentes,2024,2025,2022,2023,225,0.792902,0.644746,0.629310
779,janelas adjacentes,2013,2022,2023,2023,111,0.790876,0.634358,0.580357
347,janelas adjacentes,2023,2024,2025,2025,55,0.788406,0.621284,0.578947
215,janelas adjacentes,2025,2025,2023,2024,183,0.788362,0.616687,0.634921
675,janelas adjacentes,2017,2021,2022,2023,225,0.787444,0.635195,0.577586


,tipo,cal_ini,cal_fim,conf_ini,conf_fim,km_com_acidente_conferido,auc,auc_dentro,captura20
199,janelas adjacentes,2021,2021,2019,2020,297,0.664945,0.586771,0.411576
1227,com defasagem,2020,2025,2015,2015,238,0.664014,0.622477,0.390805
283,janelas adjacentes,2022,2022,2017,2021,745,0.661293,0.554208,0.373464
1171,com defasagem,2022,2025,2019,2019,177,0.660479,0.618747,0.338798
195,janelas adjacentes,2020,2020,2018,2019,335,0.660406,0.580997,0.363897
143,janelas adjacentes,2018,2018,2019,2020,297,0.660230,0.595028,0.408360
1099,com defasagem,2011,2013,2016,2016,162,0.657341,0.587977,0.363636
91,janelas adjacentes,2020,2020,2019,2019,177,0.654058,0.591659,0.327869
259,janelas adjacentes,2016,2016,2011,2015,1141,0.650992,0.597164,0.394461
1179,com defasagem,2011,2013,2018,2018,165,0.648229,0.581663,0.379518


gates: 5/5 PASS | registros de IA: 2 | exportados em /home/claude/results/tables


---
## Limites

Os atributos da via são os da Declaração de Rede de 2025 em todas as janelas. Cenários com janelas curtas têm poucos quilômetros com descarrilamento e oscilam mais; foram descartados os cenários com menos de 20 quilômetros atingidos na calibração ou na conferência. Os cenários se sobrepõem no tempo e não são independentes entre si: a média resume a grade, não é um intervalo de confiança.

**Declaração de uso de IA (regra da disciplina, três partes).** Ferramentas: Claude (Anthropic), usado pelo autor como assistente de programação a partir das especificações FOCUS registradas neste notebook, e Gemini do Colab quando indicado. Parte do trabalho: geração do código de cada passo e revisão do texto das células. Responsável pelo conteúdo: o autor, que definiu o problema, as fontes, as especificações, os gates e a verificação, e executou o notebook do zero antes de reportar qualquer número.